# 🎙️ TTS FastAPI & Studio Processing (Colab Edition)

**ผู้สร้าง:** SatangThevalue | **Version:** 1.5 (Pro Edition)

สมุดโน้ตนี้จะทำการโคลนโปรเจ็กต์ `tts-fastapi-service` มาไว้ใน Google Drive ของคุณโดยอัตโนมัติ เพื่อให้:
1. **ไม่ต้องโหลดโมเดลใหม่ทุกครั้ง:** โมเดลจะถูกบันทึกเก็บไว้ถาวรใน Google Drive
2. **เปิด Public URL อัตโนมัติ:** เมื่อรันเสร็จจะได้ลิงก์ `xxxx.gradio.live` ไปใช้งานที่ไหนก็ได้
3. **รองรับ n8n & AI Agent (MCP):** สามารถเรียกใช้ผ่าน API ได้ทันที

---
### ⚠️ การเตรียมตัวก่อนรัน
ไปที่แถบเมนูด้านบนเลือก **Runtime (รันไทม์)** -> **Change runtime type (เปลี่ยนประเภทสเตจ)** -> เลือก Hardware accelerator เป็น **T4 GPU** แล้วกดบันทึก

## 1. เชื่อมต่อ Google Drive และ Clone Repository
ขั้นตอนแรก ระบบจะขออนุญาตเข้าถึง Google Drive ของคุณ เพื่อสร้างโฟลเดอร์สำหรับเก็บระบบและไฟล์โมเดล

In [ ]:
import os
from google.colab import drive
import subprocess

# 1. เชื่อมต่อ Google Drive
drive.mount('/content/drive')

# 2. กำหนด Path ที่จะเก็บโปรเจ็กต์ใน Google Drive (แก้ชื่อโฟลเดอร์ได้ตามต้องการ)
PROJECT_ROOT = '/content/drive/MyDrive/AI_Voice_Studio'
REPO_URL = 'https://github.com/SatangThevalue/tts-fastapi-service.git'

print(f"\n📁 กำลังตรวจสอบพื้นที่ใน Google Drive ที่: {PROJECT_ROOT}")
if not os.path.exists(PROJECT_ROOT):
    print("กำลังโคลน Source Code จาก GitHub...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_ROOT])
else:
    print("พบโปรเจ็กต์เดิมอยู่แล้ว กำลังอัปเดตโค้ดให้เป็นเวอร์ชั่นล่าสุด...")
    subprocess.run(["git", "-C", PROJECT_ROOT, "pull"])

# 3. ย้ายพื้นที่ทำงาน (Workspace) ไปไว้ใน Google Drive
os.chdir(PROJECT_ROOT)
print("\n✅ พร้อมเข้าสู่ขั้นตอนถัดไป (Workspace ปัจจุบัน: {}) ".format(os.getcwd()))

## 2. ติดตั้ง Dependencies และดาวน์โหลด Base Models
ขั้นตอนนี้อาจใช้เวลาประมาณ 2-3 นาทีในครั้งแรก แต่ในครั้งต่อไปโมเดลต่างๆ จะถูกดึงมาจาก Google Drive ทำให้เปิดระบบได้ไวขึ้นมาก

In [ ]:
import sys

print("📦 กำลังติดตั้งไลบรารีที่จำเป็น... (อาจใช้เวลาสักครู่)")
!pip install -q fastapi uvicorn python-multipart pydantic gradio>=4.0.0 pedalboard==0.9.8 soundfile==0.12.1 mcp>=0.1.0 httpx numpy nest_asyncio

# =============================================================
# [ส่วนจำลองพื้นที่เก็บโมเดล]
# เนื่องจากปัจจุบันโค้ดใช้การ Mock-up เพื่อทดสอบระบบ UI/API
# โค้ดส่วนนี้จะสร้างโฟลเดอร์โมเดลไว้ให้ เมื่อคุณนำโค้ดของ 
# OmniVoice/CosyVoice ของจริงมาใส่ โมเดลจะถูกโหลดมาเก็บที่นี่
# =============================================================
MODEL_DIR = os.path.join(PROJECT_ROOT, "pretrained_models")
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"\n🧠 พื้นที่จัดเก็บ Base Model เตรียมพร้อมแล้วที่: {MODEL_DIR}")
print("(ระบบจะใช้โฟลเดอร์นี้เพื่อไม่ให้ Colab ต้องดาวน์โหลดโมเดล LLM/TTS ใหม่ทุกรอบ)")
print("\n✅ ติดตั้งสำเร็จ!")

## 3. สตาร์ทเซิร์ฟเวอร์ (Gradio UI + n8n API + MCP Server)
รัน Cell ด้านล่างนี้เพื่อเปิดระบบ รอสักครู่ระบบจะสร้างลิงก์ `https://xxxxxx.gradio.live` ให้คุณคลิกเข้าใช้งาน

In [ ]:
import nest_asyncio
import uvicorn
import gradio as gr
from app import app, demo

print("\n" + "="*60)
print("🌟 กำลังเปิดระบบ AI Voice Studio!")
print("กรุณารอสักครู่... ลิงก์ Public URL (gradio.live) จะปรากฏขึ้นด้านล่าง")
print("="*60 + "\n")

# แก้ปัญหา Event Loop ซ้อนทับกันของ Colab
nest_asyncio.apply()

# เปิด Web UI เบื้องหลังและสร้างลิงก์แชร์แบบ Public
demo.launch(server_name="0.0.0.0", server_port=7860, share=True, prevent_thread_lock=True)

# สตาร์ท FastAPI (ควบคุมพอร์ต 7860)
uvicorn.run(app, host="0.0.0.0", port=7860)